In [ ]:
import psutil
from functools import partial
import sys
# !{sys.executable} --version
# !{sys.executable} -m pip install shap --upgrade 
import joblib
import time
from shutil import copy
import numpy as np
import pandas as pd
#import tensorflow as tf
import os
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm
import pickle
# import xgboost as xgb

from glob import glob
# import psi4
# from helper_CC_ML_spacial import *

import pyscf
from pyscf import gto, scf, mcscf, cc

import ffsim
import numpy as np
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit, QuantumRegister
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler

from qiskit_addon_sqd.fermion import SCIResult, diagonalize_fermionic_hamiltonian

from ansatzmap import get_zigzag_physical_layout

from tqdm.notebook import tqdm
from DDLUCJ import DDLUCJ, GrabAmps 

In [ ]:
BasisDirs=glob('data/*')

In [ ]:
energyDF=pd.read_csv("../../../classical/energies.csv",index_col=0)

In [ ]:
moldf = pd.read_csv('molecules.csv')
activespacedf = pd.read_csv("active_spaces.csv")

In [ ]:
BasisSets = ['STO-3G','cc-pVDZ','aug-cc-pVDZ']

In [ ]:
# os.mkdir('jobids')

In [ ]:
# 1080 experiments
experiment = []
for row in tqdm(moldf.itertuples(),desc='Molecule'):
    moldict = row._asdict()
    name=moldict['molecule']
    n_electrons=moldict['n_electrons']
    num_orbitals=moldict['num_orbitals']
    xyzname = moldict['mol_filename']
    pathxyz = os.path.join("../../../classical/structures/",xyzname)
    
    
    
    for basis in tqdm(BasisSets,desc='Basis Set'):
        ampdict = GrabAmps(name,basis)
        for k,v in tqdm(ampdict.items(),desc="Amplitudes"):
            t1, t2 = v
            
            for L in tqdm(range(1,6),desc="Layers"):
                if os.path.exists(f"./jobids/{name}_LUCJ_L{L}_{basis}_{k}.txt")==False:
                    print(f"Running {name}_LUCJ_L{L}_{basis}_{k}")
                    initDDLUCJ = DDLUCJ(StructurePath=pathxyz, 
                                        BasisSet=basis, 
                                        NElec=n_electrons,
                                        NOrb=num_orbitals,
                                        injected=True,
                                        t1=t1, 
                                        t2=t2,
                                        n_reps = L,
                                        channel = 'ibm_quantum_platform',
                                        instance = INSTANCE,
                                        backend = None,         
                                        optimization_level=3,
                                        verbose=True)
                    
                    JobID = initDDLUCJ(postprocess=False)                
                    # initDDLUCJ.circuit.decompose(reps=2).draw('mpl',fold=-1, filename=f"./circuitdrawings/{name}_LUCJ_L{L}_{basis}_{k}.jpeg")
                    experiment.append((name,basis,k,L,JobID))
                    with open(f"./jobids/{name}_LUCJ_L{L}_{basis}_{k}.txt",'w') as f:
                        for i in (name,basis,k,L,JobID):
                            f.write(f'{i}\n') 
                else:
                    print(f"Exists: ./jobids/{name}_LUCJ_L{L}_{basis}_{k}.txt")
                            

                
# pd.DataFrame(experiment,columns=['Name','Basis',"Pairs","Layers","JobID"]).to_excel("experiments.xlsx")